# Ames Housing · el recorrido completo

**Curso:** Machine learning — aprendizaje supervisado de regresión en Python
**Solución del ejercicio de la sesión 2** · Andrés Felipe Puerta Vélez

---

En clase preparamos 93 coches con 22 columnas. Aquí son **1.460 casas con 80
columnas**, y aparecen tres cosas que Cars93 no tenía:

1. **Vacíos que no son vacíos, a gran escala.** En Cars93 era una columna
   (`AirBags`). Aquí son *quince*, y casi todas significan "esta casa no tiene
   eso".
2. **Escalas ordinales de verdad, y muchas.** Diez columnas van de `Po`
   (pobre) a `Ex` (excelente). El orden lo pone quien conoce el dominio.
3. **Variables derivadas.** Con 80 columnas empieza a rendir crear unas pocas
   nuevas: la antigüedad, la superficie total, los baños totales.

El recorrido es el mismo de la sesión. Lo que cambia es que cada decisión pesa
más, porque hay más columnas donde equivocarse.

> **Este cuaderno viene sin ejecutar.** Ames se descarga de OpenML al abrirlo,
> así que las salidas dependen de su copia de los datos. Ejecútenlo entero
> (*Run all*) y compárenlo con lo que dice cada explicación: si algo no cuadra,
> ese desajuste es el ejercicio.

## 0 · Preparación

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

In [ ]:
from sklearn.datasets import fetch_openml

ames = fetch_openml(name="house_prices", as_frame=True).frame
print(ames.shape)
ames.head()

---
## 1 · Primer contacto

Lo mismo que en clase: mirar antes de tocar. Con 81 columnas, `info()` de golpe
no se lee; conviene resumir.

In [ ]:
resumen = pd.DataFrame({
    "tipo": ames.dtypes.astype(str),
    "vacíos_%": (ames.isna().mean() * 100).round(1),
    "distintos": ames.nunique(),
})
print("filas:", len(ames), " columnas:", ames.shape[1])
resumen.sort_values("vacíos_%", ascending=False).head(20)

In [ ]:
ames["SalePrice"].describe().round(0)

---
## 2 · La decisión que ningún modelo toma

Antes de nada: **buscar columnas que no deberían estar**. En Cars93 eran
`Min.Price` y `Max.Price`, que juntas *eran* el precio. Aquí hay que revisar dos
cosas: identificadores y posibles fugas.

In [ ]:
# Id es un identificador: no es información, es el número de fila
print("¿Id tiene un valor distinto por casa?",
      ames["Id"].nunique() == len(ames))

# ¿alguna columna sospechosamente correlacionada con el precio?
numericas = ames.select_dtypes(include=np.number).drop(columns=["Id", "SalePrice"])
corr = numericas.corrwith(ames["SalePrice"]).abs().sort_values(ascending=False)
corr.head(8).round(3)

`OverallQual` y `GrLivArea` correlacionan fuerte, pero eso es **información
legítima**: la calidad general y los metros cuadrados se conocen antes de
vender. No es fuga.

La diferencia con `Min.Price` es esa: aquella columna no existiría hasta después
de saber el precio. Estas sí.

In [ ]:
X = ames.drop(columns=["Id", "SalePrice"])
y = ames["SalePrice"]
print(X.shape, y.shape)

---
## 3 · Los vacíos: el mapa

Diecinueve columnas con vacíos. Pero **no todas significan lo mismo**, y ahí
está casi todo el trabajo de este dataset.

In [ ]:
vacios = (X.isna().mean() * 100).round(1)
vacios = vacios[vacios > 0].sort_values(ascending=False)
print(vacios.to_string())

### 3.1 Los que significan "no tiene"

Si abren el diccionario de datos de Ames, `PoolQC = NA` no quiere decir "no
sabemos la calidad de la piscina": quiere decir **que la casa no tiene
piscina**. Igual con el callejón, la valla, la chimenea, el garaje y el sótano.

Es exactamente el caso de `AirBags` de la sesión, pero repetido quince veces.
Imputarlos con la moda sería inventarse piscinas.

In [ ]:
# NA = "esta casa no tiene eso". Se convierten en una categoría propia.
SIN_ESTO = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
            "GarageType", "GarageFinish", "GarageQual", "GarageCond",
            "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
            "BsmtFinType2", "MasVnrType"]

# se filtra por si alguna versión del dataset no trae alguna columna
sin_esto = [c for c in SIN_ESTO if c in X.columns]
print(len(sin_esto), "columnas donde el vacío es una categoría:")
print(sin_esto)

In [ ]:
X[sin_esto] = X[sin_esto].fillna("No_tiene")

# comprobación: ya no queda ningún vacío en esas columnas
print("vacíos restantes en ese grupo:", int(X[sin_esto].isna().sum().sum()))
X["PoolQC"].value_counts()

### 3.2 Los que sí son datos perdidos

Quedan tres, y cada uno pide algo distinto.

In [ ]:
restantes = (X.isna().mean() * 100).round(2)
print(restantes[restantes > 0].sort_values(ascending=False).to_string())

- **`LotFrontage`** (17,7 %) — los metros de fachada. Sí es un dato perdido. Y
  es *MAR*: depende del barrio, porque las parcelas de un mismo barrio se
  parecen. Lo correcto es imputar **por barrio**, no con la mediana global.
- **`GarageYrBlt`** (5,5 %) — falta justo en las casas sin garaje. Es el mismo
  caso de arriba, pero en una columna numérica: no se puede poner "No_tiene" en
  un número.
- **`MasVnrArea`** y **`Electrical`** — un puñado de filas. Mediana y moda.

In [ ]:
# LotFrontage por barrio: la mediana del propio vecindario
mediana_barrio = X.groupby("Neighborhood")["LotFrontage"].median()
print("barrios sin ni un solo dato:",
      int(mediana_barrio.isna().sum()), "de", len(mediana_barrio))

antes = X["LotFrontage"].isna().sum()
X["LotFrontage"] = X["LotFrontage"].fillna(
    X.groupby("Neighborhood")["LotFrontage"].transform("median"))
print(f"vacíos en LotFrontage: {antes} -> {X['LotFrontage'].isna().sum()}")

> ⚠️ **Ojo con la fuga.** Esta imputación por barrio la estamos haciendo
> *fuera* de la validación cruzada, igual que hicimos en clase para enseñar el
> punto. En un trabajo serio va dentro de un transformador propio. La dejamos
> aquí porque `GroupBy` no tiene un equivalente directo en scikit-learn y el
> objetivo del cuaderno es el recorrido, no la ingeniería.

In [ ]:
# GarageYrBlt: si no hay garaje, no hay año. Se marca con 0 y se añade
# una columna que diga que la casa no tiene garaje.
X["Sin_garaje"] = X["GarageYrBlt"].isna().astype(int)
X["GarageYrBlt"] = X["GarageYrBlt"].fillna(0)

print("casas sin garaje:", int(X["Sin_garaje"].sum()))
print("vacíos que quedan en todo X:", int(X.isna().sum().sum()),
      "  (MasVnrArea y Electrical, que van dentro del pipeline)")

---
## 4 · Ordinales de verdad

Diez columnas usan la misma escala de calidad. Ese orden es conocimiento de
dominio: no lo puede adivinar el código, y el alfabético lo pone al revés
(`Ex` < `Fa` < `Gd` alfabéticamente, cuando `Ex` es el mejor).

In [ ]:
CALIDAD = ["No_tiene", "Po", "Fa", "TA", "Gd", "Ex"]   # de peor a mejor

ORDINALES = {
    "ExterQual": CALIDAD[1:], "ExterCond": CALIDAD[1:],
    "HeatingQC": CALIDAD[1:], "KitchenQual": CALIDAD[1:],
    "BsmtQual": CALIDAD, "BsmtCond": CALIDAD,
    "FireplaceQu": CALIDAD, "GarageQual": CALIDAD,
    "GarageCond": CALIDAD, "PoolQC": CALIDAD,
    "BsmtExposure": ["No_tiene", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["No_tiene", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["No_tiene", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "GarageFinish": ["No_tiene", "Unf", "RFn", "Fin"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "PavedDrive": ["N", "P", "Y"],
    "CentralAir": ["N", "Y"],
}

ordinales = {c: o for c, o in ORDINALES.items() if c in X.columns}
print(len(ordinales), "columnas ordinales")

In [ ]:
# Comprobación imprescindible: ¿el orden declarado cubre lo que hay?
for col, orden in ordinales.items():
    reales = set(X[col].dropna().unique())
    sobran = reales - set(orden)
    if sobran:
        print(f"  ATENCIÓN en {col}: valores no declarados -> {sobran}")
print("comprobación terminada")

Esa comprobación no es opcional. Si un nivel se queda fuera del orden declarado,
`OrdinalEncoder` lanza un error en mitad de la validación cruzada y cuesta
entender de dónde viene.

### 4.1 La categórica disfrazada de número

`MSSubClass` son códigos: 20 es "1 planta de 1946 en adelante", 60 es "2 plantas
de 1946 en adelante". Viene como número, pero **no es una cantidad**: que 60 sea
el triple de 20 no significa nada.

In [ ]:
print(sorted(X["MSSubClass"].unique()))
X["MSSubClass"] = X["MSSubClass"].astype(str)   # a texto, para tratarla como nominal

---
## 5 · Nominales y cardinalidad

In [ ]:
nominales = [c for c in X.columns
             if c not in ordinales and X[c].dtype == object]
cardinalidad = X[nominales].nunique().sort_values(ascending=False)
print(f"{len(nominales)} columnas nominales")
cardinalidad.head(8)

In [ ]:
# ¿Cuántas columnas generaría un one-hot sin control?
print("columnas que crearía el one-hot:", int(cardinalidad.sum() - len(nominales)))
print("filas disponibles:", len(X))

Con 1.460 filas, ese número de columnas ya no es el disparate que era en Cars93
(62 columnas para 93 coches). Aun así, `Neighborhood` con 25 niveles y
`Exterior2nd` con 16 tienen categorías de muy pocas casas, así que
`min_frequency` sigue mereciendo la pena.

---
## 6 · Variables derivadas

Con 80 columnas empieza a rentar crear unas pocas. Estas tres salen del sentido
común de quien compra una casa, no de mirar los datos.

In [ ]:
X["Antiguedad"]      = X["YrSold"] - X["YearBuilt"]
X["Anos_reformada"]  = X["YrSold"] - X["YearRemodAdd"]
X["Superficie_total"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
X["Banos_totales"]   = (X["FullBath"] + 0.5 * X["HalfBath"]
                        + X["BsmtFullBath"] + 0.5 * X["BsmtHalfBath"])

derivadas = ["Antiguedad", "Anos_reformada", "Superficie_total", "Banos_totales"]
X[derivadas].describe().round(1)

In [ ]:
# ¿aportan? se mira su correlación con el precio
X[derivadas].corrwith(y).round(3)

---
## 7 · El objetivo está sesgado

Como el precio de los diamantes en clase: cola larga a la derecha.

In [ ]:
print(f"asimetría de SalePrice      : {y.skew():.3f}")
print(f"asimetría de log(SalePrice) : {np.log1p(y).skew():.3f}")
y.describe().round(0)

La competencia original de Kaggle se evalúa con el **error cuadrático medio
sobre el logaritmo del precio**, justo por esto: equivocarse en 20.000 dólares
pesa distinto en una casa de 100.000 que en una de 500.000.

---
## 8 · El pipeline completo

Tres ramas, una por tipo de columna.

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
ord_cols = list(ordinales.keys())
nom_cols = [c for c in X.columns if c not in num_cols and c not in ord_cols]

print(f"numéricas {len(num_cols)} · ordinales {len(ord_cols)} · nominales {len(nom_cols)}")
assert len(num_cols) + len(ord_cols) + len(nom_cols) == X.shape[1]

In [ ]:
rama_num = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("esc", StandardScaler()),
])

rama_ord = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=[ordinales[c] for c in ord_cols],
                           handle_unknown="use_encoded_value",
                           unknown_value=-1)),
    ("esc", StandardScaler()),
])

rama_nom = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first",
                          min_frequency=20)),
])

pre = ColumnTransformer([
    ("num", rama_num, num_cols),
    ("ord", rama_ord, ord_cols),
    ("nom", rama_nom, nom_cols),
])

print("columnas tras el preprocesamiento:", pre.fit(X).transform(X).shape[1])

---
## 9 · Los chequeos, como en clase

El mismo banco de pruebas: dos modelos fijos, validación cruzada de 5 pliegues,
MSE y R². Sobre el logaritmo del precio, que es la métrica del problema.

In [ ]:
historial = []

def chequeo(etapa, pipeline, Xu, yu):
    r = cross_validate(pipeline, Xu, yu, cv=5, error_score="raise",
                       scoring=["neg_root_mean_squared_error", "r2"])
    historial.append([etapa,
                      round(-r["test_neg_root_mean_squared_error"].mean(), 0),
                      round(r["test_r2"].mean(), 4)])
    return pd.DataFrame(historial, columns=["etapa", "RMSE (USD)", "R2"])

def con_log(modelo, preproceso=None):
    """Predice el logaritmo del precio y deshace la transformación."""
    interior = (modelo if preproceso is None
                else Pipeline([("pre", preproceso), ("reg", modelo)]))
    return TransformedTargetRegressor(regressor=interior,
                                      func=np.log1p, inverse_func=np.expm1)

In [ ]:
# Etapa 1 · solo lo que ya es número, y fuera las filas con vacíos
X_crudo = X[num_cols].dropna()
chequeo("1 · numéricas + dropna",
        con_log(LinearRegression()), X_crudo, y.loc[X_crudo.index])

In [ ]:
# Etapa 2 · + imputación y escalado de las numéricas
chequeo("2 · + imputación y escalado",
        con_log(LinearRegression(), rama_num), X[num_cols], y)

In [ ]:
# Etapa 3 · + las ordinales, con su orden declarado
pre_con_ord = ColumnTransformer([("num", rama_num, num_cols),
                                 ("ord", rama_ord, ord_cols)])
chequeo("3 · + ordinales", con_log(LinearRegression(), pre_con_ord), X, y)

In [ ]:
# Etapa 4 · + las nominales: el pipeline completo
chequeo("4 · + nominales (completa)",
        con_log(LinearRegression(), pre), X, y)

---
## 10 · Y ahora sí, afinar

Con los datos preparados, se prueban modelos. La búsqueda va **dentro** de la
validación cruzada, como vimos en clase.

In [ ]:
chequeo("5 · Ridge con alpha buscado",
        con_log(GridSearchCV(Ridge(), {"alpha": [1, 10, 30, 100]},
                             cv=5, scoring="r2"), pre),
        X, y)

In [ ]:
# HistGradientBoosting: se come los NaN y las categorías, así que se le da
# el X con las columnas de texto marcadas como "category" y nada más
X_arbol = X.copy()
for c in ord_cols + nom_cols:
    X_arbol[c] = X_arbol[c].astype("category")

chequeo("6 · HistGradientBoosting",
        con_log(GridSearchCV(
            HistGradientBoostingRegressor(random_state=7,
                                          categorical_features="from_dtype"),
            {"max_leaf_nodes": [15, 31], "learning_rate": [0.05, 0.1]},
            cv=5, scoring="r2")),
        X_arbol, y)

---
## 11 · La foto completa

In [ ]:
pd.DataFrame(historial, columns=["etapa", "RMSE (USD)", "R2"])

### Cómo leer esta tabla

- **El RMSE está en dólares.** Como `TransformedTargetRegressor` deshace el
  logaritmo antes de predecir, el error sale en la escala original y se lee
  directamente: "nos equivocamos, de media, en tantos dólares".
- **Kaggle usa otra métrica:** el RMSE sobre el *logaritmo* del precio. Para
  reproducirla basta con entrenar contra `np.log1p(y)` y no deshacer la
  transformación, midiendo entonces contra `np.log1p(y)`.
- **Cada etapa añade un tipo de columna**, igual que en clase. Si alguna no
  mejora, eso también es información: quiere decir que esa familia de columnas
  no aportaba en este problema.

### Lo que Ames enseña y Cars93 no podía

1. **Los vacíos son un problema de dominio, no de código.** Quince columnas
   donde `NaN` significaba "no tiene". Ningún imputador automático lo habría
   acertado, y el diccionario de datos valía más que cualquier técnica.
2. **Las ordinales hay que declararlas a mano, y comprobarlas.** Diez escalas
   `Po`→`Ex`, más otras nueve con órdenes propios. La celda que comprueba que no
   falte ningún nivel evita un error muy molesto de depurar.
3. **Una columna numérica puede no ser una cantidad.** `MSSubClass` son códigos.
4. **Con muchas columnas, las derivadas empiezan a rentar.** Antigüedad,
   superficie total y baños totales salen del sentido común, no de los datos.

---
## 12 · Para seguir

- **Imputar `LotFrontage` dentro del pipeline**, con un transformador propio,
  para quitar la fuga que dejamos a propósito en la sección 3.2.
- **Probar target encoding en `Neighborhood`**, con el suavizado y la validación
  cruzada interna que mencionamos en clase.
- **Mirar los atípicos:** hay casas con `GrLivArea` enorme y precio bajo que son
  ventas entre familiares. El propio autor del dataset recomienda quitarlas.
- **Comparar contra el ranking público de Kaggle** para ver dónde queda su
  RMSE.